<a href="https://colab.research.google.com/github/NoeliaFerrero/Comi-96160-Data-Science-II-Machine-Learning-para-la-Ciencia-de-Datos-Diplomaturas/blob/main/Semana%205%20Apis%20y%20Data%20Wrangling/APIs_Data_Wrangling_Inicial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧩 APIs + Data Wrangling con Pandas

**Nivel:** Inicial — Data Analytics / Data Science  
**Modalidad:** Google Colab  
**Objetivo:** obtener datos desde una API, transformarlos en DataFrames y combinarlos para responder preguntas de negocio.

### El recorrido de hoy

```text
🌎 API
  ↓
📦 JSON
  ↓
🐼 DataFrame
  ↓
🧹 Data Wrangling
  ↓
🔗 concat() / merge()
  ↓
📊 Análisis
```

> **Idea clave:** no estamos aprendiendo funciones aisladas. Estamos construyendo un pequeño flujo de trabajo de datos.

## 🎯 Objetivos

Al finalizar la clase vas a poder:

- explicar con tus palabras qué es una API;
- realizar una consulta a una API desde Python;
- interpretar una respuesta JSON;
- convertir datos JSON en un DataFrame;
- aplicar transformaciones básicas de Data Wrangling;
- distinguir `concat()` de `merge()`;
- combinar datos de distintas fuentes para responder una pregunta.

### Regla de oro

**`concat()` apila datos. `merge()` relaciona datos.**

# 1. ¿Qué es una API?

Una **API (Application Programming Interface)** permite que dos aplicaciones se comuniquen.

Pensala como una puerta:

```text
👩‍💻 Python
   │
   │ "Necesito estos datos"
   ▼
🚪 API
   │
   ▼
🌎 Servicio externo
   │
   ▼
📦 JSON
```

En vez de copiar datos manualmente desde una página web, podemos pedirlos mediante código.

In [2]:
# Vamos a utilizar requests para hacer una consulta HTTP.
import requests
import pandas as pd

print("requests y pandas importados correctamente")

requests y pandas importados correctamente


# 2. Consumimos una API real 🌤️

Vamos a usar **Open-Meteo**, una API pública de datos meteorológicos que no requiere API Key.

Para el ejemplo vamos a consultar Córdoba:

- Latitud: `-31.42`
- Longitud: `-64.18`

La URL base será:

`https://api.open-meteo.com/v1/forecast`

In [3]:
url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": -31.42,
    "longitude": -64.18,
    "current": "temperature_2m,relative_humidity_2m,wind_speed_10m",
    "timezone": "America/Argentina/Cordoba"
}

respuesta = requests.get(url, params=params, timeout=20)

print("Status code:", respuesta.status_code)
print("URL consultada:", respuesta.url)

Status code: 200
URL consultada: https://api.open-meteo.com/v1/forecast?latitude=-31.42&longitude=-64.18&current=temperature_2m%2Crelative_humidity_2m%2Cwind_speed_10m&timezone=America%2FArgentina%2FCordoba


### ¿Qué significa el `status_code`?

Algunos códigos frecuentes:

| Código | Significado |
|---:|---|
| `200` | Todo OK |
| `400` | La solicitud tiene un problema |
| `401` | Falta autorización |
| `404` | Recurso no encontrado |
| `500` | Error del servidor |

Si obtenemos `200`, podemos continuar.

In [4]:
# La respuesta llega en formato JSON.
data = respuesta.json()

type(data), data.keys()

(dict,
 dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'current_units', 'current']))

# 3. Del JSON al DataFrame

Un JSON puede contener información anidada.

Vamos a mirar primero la respuesta:

```python
data
```

y luego localizar la sección `current`, que contiene las mediciones actuales.

In [5]:
# Inspeccionamos la sección "current"
data["current"]

{'time': '2026-08-10T10:45',
 'interval': 900,
 'temperature_2m': 7.4,
 'relative_humidity_2m': 57,
 'wind_speed_10m': 4.2}

In [6]:
# Convertimos el diccionario "current" en un DataFrame de una fila.
clima = pd.DataFrame([data["current"]])

clima

,time,interval,temperature_2m,relative_humidity_2m,wind_speed_10m
0,2026-08-10T10:45,900,7.4,57,4.2


## 🧠 Pregunta rápida

¿Por qué usamos:

```python
pd.DataFrame([data["current"]])
```

y no simplemente:

```python
pd.DataFrame(data["current"])
```

Porque queremos convertir **un registro** en una fila. La lista `[...]` representa una colección de registros.

In [7]:
# Algunas primeras exploraciones
print("Filas y columnas:", clima.shape)
print("\nColumnas:")
print(clima.columns.tolist())

print("\nTipos:")
display(clima.dtypes)

print("\nPrimeras filas:")
display(clima.head())

Filas y columnas: (1, 5)

Columnas:
['time', 'interval', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m']

Tipos:


,0
time,object
interval,int64
temperature_2m,float64
relative_humidity_2m,int64
wind_speed_10m,float64



Primeras filas:


,time,interval,temperature_2m,relative_humidity_2m,wind_speed_10m
0,2026-08-10T10:45,900,7.4,57,4.2


# 4. Data Wrangling 🧹

**Data Wrangling** es el proceso de preparar los datos para poder analizarlos.

Puede incluir:

- cambiar nombres de columnas;
- cambiar tipos;
- tratar valores faltantes;
- crear nuevas variables;
- filtrar registros;
- combinar DataFrames.

No hay una única receta: depende del problema.

In [8]:
# Renombramos algunas columnas para que sean más amigables.
clima = clima.rename(columns={
    "time": "fecha_hora",
    "temperature_2m": "temperatura_c",
    "relative_humidity_2m": "humedad_pct",
    "wind_speed_10m": "viento_kmh"
})

clima

,fecha_hora,interval,temperatura_c,humedad_pct,viento_kmh
0,2026-08-10T10:45,900,7.4,57,4.2


In [9]:
# Convertimos la fecha y creamos una variable derivada.
clima["fecha_hora"] = pd.to_datetime(clima["fecha_hora"])
clima["temperatura_f"] = clima["temperatura_c"] * 9/5 + 32

clima

,fecha_hora,interval,temperatura_c,humedad_pct,viento_kmh,temperatura_f
0,2026-08-10 10:45:00,900,7.4,57,4.2,45.32


### ¿Qué hicimos?

Transformamos:

```text
temperature_2m
```

en:

```text
temperatura_c
```

y además creamos una nueva variable:

```text
temperatura_f
```

Eso también es Data Wrangling.

# 5. Trabajamos con datos de negocio

Ahora vamos a crear dos DataFrames pequeños.

Imaginemos que una empresa tiene ventas registradas en dos meses.

In [10]:
ventas_enero = pd.DataFrame({
    "id_cliente": [101, 102, 103, 104],
    "mes": ["Enero"] * 4,
    "monto": [120000, 85000, 150000, 92000]
})

ventas_febrero = pd.DataFrame({
    "id_cliente": [102, 103, 104, 105],
    "mes": ["Febrero"] * 4,
    "monto": [110000, 170000, 98000, 125000]
})

display(ventas_enero)
display(ventas_febrero)

,id_cliente,mes,monto
0,101,Enero,120000
1,102,Enero,85000
2,103,Enero,150000
3,104,Enero,92000


,id_cliente,mes,monto
0,102,Febrero,110000
1,103,Febrero,170000
2,104,Febrero,98000
3,105,Febrero,125000


# 6. `concat()` → apilar

Tenemos dos DataFrames con la **misma estructura**:

```text
ENERO
─────
101
102
103
104

FEBRERO
───────
102
103
104
105
```

Queremos poner uno debajo del otro.

### Usamos:

```python
pd.concat([...])
```

In [11]:
ventas = pd.concat(
    [ventas_enero, ventas_febrero],
    ignore_index=True
)

ventas

,id_cliente,mes,monto
0,101,Enero,120000
1,102,Enero,85000
2,103,Enero,150000
3,104,Enero,92000
4,102,Febrero,110000
5,103,Febrero,170000
6,104,Febrero,98000
7,105,Febrero,125000


### 🧠 Regla mental

**`concat()` = apilar**

```text
┌──────────┐
│  Enero   │
├──────────┤
│ Febrero  │
└──────────┘
```

Es parecido a un `UNION ALL` en SQL.

# 7. `merge()` → relacionar 🔗

Ahora tenemos información de clientes en otro DataFrame.

```text
clientes
---------
id_cliente
nombre
ciudad
```

y:

```text
ventas
------
id_cliente
mes
monto
```

Ambos comparten una clave:

**`id_cliente`**

Eso nos permite relacionarlos.

In [12]:
clientes = pd.DataFrame({
    "id_cliente": [101, 102, 103, 104, 105, 106],
    "nombre": ["Ana", "Bruno", "Carla", "Diego", "Elena", "Fabián"],
    "ciudad": ["Córdoba", "Rosario", "Córdoba", "Mendoza", "Córdoba", "Salta"]
})

clientes

,id_cliente,nombre,ciudad
0,101,Ana,Córdoba
1,102,Bruno,Rosario
2,103,Carla,Córdoba
3,104,Diego,Mendoza
4,105,Elena,Córdoba
5,106,Fabián,Salta


In [13]:
ventas_completas = ventas.merge(
    clientes,
    on="id_cliente",
    how="left"
)

ventas_completas

,id_cliente,mes,monto,nombre,ciudad
0,101,Enero,120000,Ana,Córdoba
1,102,Enero,85000,Bruno,Rosario
2,103,Enero,150000,Carla,Córdoba
3,104,Enero,92000,Diego,Mendoza
4,102,Febrero,110000,Bruno,Rosario
5,103,Febrero,170000,Carla,Córdoba
6,104,Febrero,98000,Diego,Mendoza
7,105,Febrero,125000,Elena,Córdoba


### ¿Qué pasó con el cliente 105?

Apareció en febrero y `merge()` buscó su información en `clientes`.

### ¿Y el cliente 106?

Está en `clientes`, pero no aparece en `ventas`.

Por eso **no aparece** en nuestro resultado: hicimos un `left merge` tomando `ventas` como DataFrame de la izquierda.

# 8. `merge()` y los tipos de JOIN

Los mismos conceptos aparecen en SQL:

| Pandas | SQL |
|---|---|
| `merge(..., how="inner")` | `INNER JOIN` |
| `merge(..., how="left")` | `LEFT JOIN` |
| `merge(..., how="right")` | `RIGHT JOIN` |
| `merge(..., how="outer")` | `FULL OUTER JOIN` |

### Regla mental

**`merge()` = relacionar por una clave.**

In [14]:
# Probemos un LEFT JOIN y un INNER JOIN

left_join = ventas.merge(clientes, on="id_cliente", how="left")
inner_join = ventas.merge(clientes, on="id_cliente", how="inner")

print("LEFT JOIN:")
display(left_join)

print("INNER JOIN:")
display(inner_join)

LEFT JOIN:


,id_cliente,mes,monto,nombre,ciudad
0,101,Enero,120000,Ana,Córdoba
1,102,Enero,85000,Bruno,Rosario
2,103,Enero,150000,Carla,Córdoba
3,104,Enero,92000,Diego,Mendoza
4,102,Febrero,110000,Bruno,Rosario
5,103,Febrero,170000,Carla,Córdoba
6,104,Febrero,98000,Diego,Mendoza
7,105,Febrero,125000,Elena,Córdoba


INNER JOIN:


,id_cliente,mes,monto,nombre,ciudad
0,101,Enero,120000,Ana,Córdoba
1,102,Enero,85000,Bruno,Rosario
2,103,Enero,150000,Carla,Córdoba
3,104,Enero,92000,Diego,Mendoza
4,102,Febrero,110000,Bruno,Rosario
5,103,Febrero,170000,Carla,Córdoba
6,104,Febrero,98000,Diego,Mendoza
7,105,Febrero,125000,Elena,Córdoba


# 9. Mini análisis 📊

Ahora que tenemos un DataFrame completo, podemos responder preguntas.

### ¿Cuánto vendió cada cliente?

In [15]:
resumen_clientes = (
    ventas_completas
    .groupby(["id_cliente", "nombre", "ciudad"], as_index=False)["monto"]
    .sum()
    .sort_values("monto", ascending=False)
)

resumen_clientes

,id_cliente,nombre,ciudad,monto
2,103,Carla,Córdoba,320000
1,102,Bruno,Rosario,195000
3,104,Diego,Mendoza,190000
4,105,Elena,Córdoba,125000
0,101,Ana,Córdoba,120000


In [16]:
# ¿Qué ciudad tuvo mayor facturación?
resumen_ciudades = (
    ventas_completas
    .groupby("ciudad", as_index=False)["monto"]
    .sum()
    .sort_values("monto", ascending=False)
)

resumen_ciudades

,ciudad,monto
0,Córdoba,565000
2,Rosario,195000
1,Mendoza,190000


# 🧠 Antes de terminar...

Completemos estas frases:

### API
> Una API permite __________________________.

### JSON
> Una respuesta de una API puede venir en formato __________________.

### Data Wrangling
> Es el proceso de __________________________.

### `concat()`
> Lo usamos cuando queremos __________________________.

### `merge()`
> Lo usamos cuando queremos __________________________.

Si podés explicar estas cinco ideas, ya tenés lo fundamental de la clase.

# 🚀 DESAFÍO FINAL — Equipo de DS

Ahora les toca a ustedes.

## Situación

Una empresa quiere analizar sus ventas de enero y febrero.

Ya tenemos:

- `ventas_enero`
- `ventas_febrero`
- `clientes`

### Objetivos

1. Combinar enero y febrero en un único DataFrame.
2. Incorporar nombre y ciudad de cada cliente.
3. Calcular cuánto vendió cada cliente.
4. Identificar qué ciudad generó mayor facturación.
5. Identificar clientes que **no realizaron ninguna compra**.

### Restricciones

Deben utilizar:

- `pd.concat()`
- `merge()`
- `groupby()`

### ⭐ Bonus

Crear una columna:

```text
nivel_cliente
```

con:

- `"Alto"` si vendió >= 250000
- `"Medio"` si vendió entre 150000 y 249999
- `"Bajo"` si vendió < 150000

No miren la solución inmediatamente. Primero intenten resolverlo ustedes.

In [ ]:
# ============================================
# 🚀 DESAFÍO FINAL — RESOLUCIÓN
# ============================================

# 1. Combinar los meses
ventas = pd.concat(
    [ventas_enero, ventas_febrero],
    ignore_index=True
)

# 2. Incorporar datos de clientes
ventas_completas = ventas.merge(
    clientes,
    on="id_cliente",
    how="left"
)

# 3. Total vendido por cliente
resumen = (
    ventas_completas
    .groupby(["id_cliente", "nombre", "ciudad"], as_index=False)["monto"]
    .sum()
    .rename(columns={"monto": "ventas_totales"})
)

# 4. Clasificación del cliente
resumen["nivel_cliente"] = resumen["ventas_totales"].apply(
    lambda x: "Alto" if x >= 250000
    else "Medio" if x >= 150000
    else "Bajo"
)

print("Resumen de clientes:")
display(resumen.sort_values("ventas_totales", ascending=False))

# 5. Facturación por ciudad
print("Facturación por ciudad:")
display(
    resumen.groupby("ciudad", as_index=False)["ventas_totales"]
    .sum()
    .sort_values("ventas_totales", ascending=False)
)

# 6. Clientes sin compras
clientes_sin_compras = clientes[
    ~clientes["id_cliente"].isin(ventas["id_cliente"])
]

print("Clientes sin compras:")
display(clientes_sin_compras)

# 🎉 Cierre

Hoy hicimos un pequeño pipeline de datos:

```text
API
 ↓
JSON
 ↓
DataFrame
 ↓
Data Wrangling
 ↓
concat()
 ↓
merge()
 ↓
groupby()
 ↓
Información para tomar decisiones
```

### La idea importante

Un DS no solamente "hace código".

**Obtiene datos → los entiende → los transforma → los combina → responde preguntas.**